# SAP HANA Hospital OPD Analytics Notebook

This notebook shows how to use **Python** with **SAP HANA** through `hdbcli` to create the schema, tables, sample data, reporting views, and business queries for the Hospital OPD Appointment and Billing Analytics problem.

## Before you run

Install the SAP HANA Python client first:

```python
!pip install hdbcli
```

Then update the connection values in the next code cell.

In [13]:
!pip install hdbcli

In [14]:
from hdbcli import dbapi
import pandas as pd

## Connection setup

Replace the placeholders with your SAP HANA details.
For SAP HANA connections in Python, `hdbcli.dbapi.connect(...)` is the standard approach, using host or address, port, user, and password. [web:27][web:34]

In [15]:
HANA_HOST = "13b7c15d-848f-40b5-9259-c9c36ab85f56.hna1.prod-eu10.hanacloud.ondemand.com"
HANA_PORT = 443
HANA_USER = "GE336872"
HANA_PASSWORD = "ObxiYJJCSh1!"

conn = dbapi.connect(
    address=HANA_HOST,
    port=HANA_PORT,
    user=HANA_USER,
    password=HANA_PASSWORD,
    encrypt=True,
    sslValidateCertificate=False  # For training/demo. Use certificate validation in production.
)
cursor = conn.cursor()
print("Connected successfully to SAP HANA Cloud")


Connected successfully to SAP HANA Cloud


## Helper functions

SAP HANA Python execution sends one SQL command at a time through `cursor.execute(...)`, so a helper function makes notebook usage cleaner. [web:40][web:30]

In [16]:
def run_sql(sql):
    cursor.execute(sql)


def fetch_df(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

## Step 1: Create schema

SAP HANA supports `CREATE SCHEMA` and `SET SCHEMA`, so we create the assignment schema and switch the session to it. [web:24][web:17]

In [17]:
try:
    # Attempting to use the user's personal schema which typically has higher privileges
    run_sql(f"SET SCHEMA {HANA_USER}")
    print(f"Schema set to {HANA_USER}")
except Exception as e:
    print("Error setting schema:", e)

Schema set to GE336872


## Step 2: Create tables

`CREATE COLUMN TABLE` is commonly used in SAP HANA examples and is appropriate for analytical use cases. [page:1]

In [18]:
tables_to_drop = ['BILLING', 'APPOINTMENTS', 'DOCTORS', 'DEPARTMENTS', 'PATIENTS']

for table in tables_to_drop:
    try:
        run_sql(f"DROP TABLE {table}")
    except Exception:
        pass

run_sql("""
CREATE COLUMN TABLE PATIENTS (
    PATIENT_ID      INTEGER PRIMARY KEY,
    PATIENT_NAME    NVARCHAR(100) NOT NULL,
    GENDER          NVARCHAR(10) NOT NULL,
    AGE             INTEGER NOT NULL,
    CITY            NVARCHAR(50) NOT NULL,
    MOBILE_NUMBER   NVARCHAR(15) NOT NULL UNIQUE
)
""")

run_sql("""
CREATE COLUMN TABLE DEPARTMENTS (
    DEPARTMENT_ID    INTEGER PRIMARY KEY,
    DEPARTMENT_NAME  NVARCHAR(100) NOT NULL UNIQUE,
    FLOOR_NUMBER     INTEGER NOT NULL
)
""")

run_sql("""
CREATE COLUMN TABLE DOCTORS (
    DOCTOR_ID         INTEGER PRIMARY KEY,
    DOCTOR_NAME       NVARCHAR(100) NOT NULL,
    DEPARTMENT_ID     INTEGER NOT NULL,
    SPECIALIZATION    NVARCHAR(100) NOT NULL,
    CONSULTATION_FEE  DECIMAL(10,2) NOT NULL,
    FOREIGN KEY (DEPARTMENT_ID) REFERENCES DEPARTMENTS(DEPARTMENT_ID)
)
""")

run_sql("""
CREATE COLUMN TABLE APPOINTMENTS (
    APPOINTMENT_ID      INTEGER PRIMARY KEY,
    PATIENT_ID          INTEGER NOT NULL,
    DOCTOR_ID           INTEGER NOT NULL,
    APPOINTMENT_DATE    DATE NOT NULL,
    APPOINTMENT_TIME    TIME NOT NULL,
    APPOINTMENT_STATUS  NVARCHAR(20) NOT NULL,
    FOREIGN KEY (PATIENT_ID) REFERENCES PATIENTS(PATIENT_ID),
    FOREIGN KEY (DOCTOR_ID) REFERENCES DOCTORS(DOCTOR_ID),
    CONSTRAINT CHK_APPOINTMENT_STATUS
        CHECK (APPOINTMENT_STATUS IN ('Completed', 'Pending', 'Cancelled'))
)
""")

run_sql("""
CREATE COLUMN TABLE BILLING (
    BILL_ID          INTEGER PRIMARY KEY,
    APPOINTMENT_ID   INTEGER NOT NULL UNIQUE,
    BILL_AMOUNT      DECIMAL(10,2) NOT NULL,
    PAYMENT_MODE     NVARCHAR(20) NOT NULL,
    PAYMENT_STATUS   NVARCHAR(20) NOT NULL,
    FOREIGN KEY (APPOINTMENT_ID) REFERENCES APPOINTMENTS(APPOINTMENT_ID),
    CONSTRAINT CHK_PAYMENT_MODE
        CHECK (PAYMENT_MODE IN ('Cash', 'UPI', 'Card', 'Insurance')),
    CONSTRAINT CHK_PAYMENT_STATUS
        CHECK (PAYMENT_STATUS IN ('Paid', 'Unpaid', 'Refunded'))
)
""")

print("All tables created successfully")

All tables created successfully


## Step 3: Insert sample data

The following cells load sample records so the views and analytics queries can be tested immediately.

In [19]:
patients = [
    (1, 'Amit Sharma', 'Male', 34, 'Delhi', '9876500001'),
    (2, 'Neha Verma', 'Female', 29, 'Noida', '9876500002'),
    (3, 'Rohan Gupta', 'Male', 41, 'Ghaziabad', '9876500003'),
    (4, 'Priya Singh', 'Female', 36, 'Delhi', '9876500004'),
    (5, 'Karan Mehta', 'Male', 52, 'Faridabad', '9876500005'),
    (6, 'Sneha Joshi', 'Female', 24, 'Ghaziabad', '9876500006')
]

departments = [
    (101, 'Cardiology', 2),
    (102, 'Orthopedics', 3),
    (103, 'General Medicine', 1),
    (104, 'Pediatrics', 4)
]

doctors = [
    (1001, 'Dr. Arvind Rao', 101, 'Cardiologist', 800.00),
    (1002, 'Dr. Meera Nair', 102, 'Orthopedic Specialist', 700.00),
    (1003, 'Dr. Sandeep Khanna', 103, 'Physician', 500.00),
    (1004, 'Dr. Pooja Bansal', 104, 'Pediatrician', 600.00),
    (1005, 'Dr. Vivek Malhotra', 101, 'Heart Specialist', 900.00)
]

appointments = [
    (5001, 1, 1001, '2026-06-25', '10:00:00', 'Completed'),
    (5002, 2, 1002, '2026-06-25', '11:00:00', 'Completed'),
    (5003, 3, 1003, '2026-06-26', '09:30:00', 'Pending'),
    (5004, 4, 1004, '2026-06-26', '12:00:00', 'Cancelled'),
    (5005, 5, 1005, '2026-06-27', '14:00:00', 'Completed'),
    (5006, 6, 1003, '2026-06-27', '15:30:00', 'Completed'),
    (5007, 1, 1002, '2026-06-28', '10:30:00', 'Pending'),
    (5008, 2, 1001, '2026-06-28', '16:00:00', 'Completed')
]

billing = [
    (9001, 5001, 800.00, 'UPI', 'Paid'),
    (9002, 5002, 700.00, 'Card', 'Paid'),
    (9003, 5003, 500.00, 'Cash', 'Unpaid'),
    (9004, 5004, 600.00, 'Cash', 'Refunded'),
    (9005, 5005, 900.00, 'Insurance', 'Paid'),
    (9006, 5006, 500.00, 'UPI', 'Paid'),
    (9007, 5007, 700.00, 'Card', 'Unpaid'),
    (9008, 5008, 800.00, 'Cash', 'Paid')
]

In [20]:
for row in patients:
    cursor.execute("INSERT INTO PATIENTS VALUES (?, ?, ?, ?, ?, ?)", row)

for row in departments:
    cursor.execute("INSERT INTO DEPARTMENTS VALUES (?, ?, ?)", row)

for row in doctors:
    cursor.execute("INSERT INTO DOCTORS VALUES (?, ?, ?, ?, ?)", row)

for row in appointments:
    cursor.execute("INSERT INTO APPOINTMENTS VALUES (?, ?, ?, ?, ?, ?)", row)

for row in billing:
    cursor.execute("INSERT INTO BILLING VALUES (?, ?, ?, ?, ?)", row)

print("Sample data inserted")

Sample data inserted


## Step 4: Create the main analytics view

SAP HANA supports `CREATE OR REPLACE VIEW`, which is useful when you need to rerun notebook cells during development. [web:15]

In [21]:
run_sql("""
CREATE OR REPLACE VIEW V_OPD_APPOINTMENT_ANALYTICS AS
SELECT
    A.APPOINTMENT_ID,
    A.APPOINTMENT_DATE,
    A.APPOINTMENT_TIME,
    P.PATIENT_NAME,
    P.CITY AS PATIENT_CITY,
    D.DOCTOR_NAME,
    DP.DEPARTMENT_NAME,
    D.SPECIALIZATION,
    D.CONSULTATION_FEE,
    B.BILL_AMOUNT,
    B.PAYMENT_MODE,
    B.PAYMENT_STATUS,
    A.APPOINTMENT_STATUS
FROM APPOINTMENTS A
INNER JOIN PATIENTS P ON A.PATIENT_ID = P.PATIENT_ID
INNER JOIN DOCTORS D ON A.DOCTOR_ID = D.DOCTOR_ID
INNER JOIN DEPARTMENTS DP ON D.DEPARTMENT_ID = DP.DEPARTMENT_ID
LEFT JOIN BILLING B ON A.APPOINTMENT_ID = B.APPOINTMENT_ID
""")

print("View V_OPD_APPOINTMENT_ANALYTICS created")

View V_OPD_APPOINTMENT_ANALYTICS created


## Step 5: Create the final assignment view

This view summarizes daily revenue by department, limited to completed appointments and paid bills.

In [22]:
run_sql("""
CREATE OR REPLACE VIEW V_DEPARTMENT_DAILY_REVENUE AS
SELECT
    A.APPOINTMENT_DATE,
    DP.DEPARTMENT_NAME,
    COUNT(A.APPOINTMENT_ID) AS TOTAL_APPOINTMENTS,
    COUNT(B.BILL_ID) AS TOTAL_PAID_BILLS,
    SUM(B.BILL_AMOUNT) AS TOTAL_REVENUE
FROM APPOINTMENTS A
INNER JOIN DOCTORS D ON A.DOCTOR_ID = D.DOCTOR_ID
INNER JOIN DEPARTMENTS DP ON D.DEPARTMENT_ID = DP.DEPARTMENT_ID
INNER JOIN BILLING B ON A.APPOINTMENT_ID = B.APPOINTMENT_ID
WHERE A.APPOINTMENT_STATUS = 'Completed'
  AND B.PAYMENT_STATUS = 'Paid'
GROUP BY A.APPOINTMENT_DATE, DP.DEPARTMENT_NAME
""")

print("View V_DEPARTMENT_DAILY_REVENUE created")

View V_DEPARTMENT_DAILY_REVENUE created


## Step 6: Validate the main view

This confirms that the full OPD analytics view is working correctly.

In [23]:
fetch_df("""
SELECT *
FROM V_OPD_APPOINTMENT_ANALYTICS
ORDER BY APPOINTMENT_DATE, APPOINTMENT_TIME
""")

,APPOINTMENT_ID,APPOINTMENT_DATE,APPOINTMENT_TIME,PATIENT_NAME,PATIENT_CITY,DOCTOR_NAME,DEPARTMENT_NAME,SPECIALIZATION,CONSULTATION_FEE,BILL_AMOUNT,PAYMENT_MODE,PAYMENT_STATUS,APPOINTMENT_STATUS
0,5001,2026-06-25,10:00:00,Amit Sharma,Delhi,Dr. Arvind Rao,Cardiology,Cardiologist,800,800,UPI,Paid,Completed
1,5002,2026-06-25,11:00:00,Neha Verma,Noida,Dr. Meera Nair,Orthopedics,Orthopedic Specialist,700,700,Card,Paid,Completed
2,5003,2026-06-26,09:30:00,Rohan Gupta,Ghaziabad,Dr. Sandeep Khanna,General Medicine,Physician,500,500,Cash,Unpaid,Pending
3,5004,2026-06-26,12:00:00,Priya Singh,Delhi,Dr. Pooja Bansal,Pediatrics,Pediatrician,600,600,Cash,Refunded,Cancelled
4,5005,2026-06-27,14:00:00,Karan Mehta,Faridabad,Dr. Vivek Malhotra,Cardiology,Heart Specialist,900,900,Insurance,Paid,Completed
5,5006,2026-06-27,15:30:00,Sneha Joshi,Ghaziabad,Dr. Sandeep Khanna,General Medicine,Physician,500,500,UPI,Paid,Completed
6,5007,2026-06-28,10:30:00,Amit Sharma,Delhi,Dr. Meera Nair,Orthopedics,Orthopedic Specialist,700,700,Card,Unpaid,Pending
7,5008,2026-06-28,16:00:00,Neha Verma,Noida,Dr. Arvind Rao,Cardiology,Cardiologist,800,800,Cash,Paid,Completed


## Step 7: Business question 1

Find all completed OPD appointments.

In [24]:
fetch_df("""
SELECT
    APPOINTMENT_ID,
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    APPOINTMENT_DATE,
    BILL_AMOUNT,
    PAYMENT_STATUS
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE APPOINTMENT_STATUS = 'Completed'
""")

,APPOINTMENT_ID,PATIENT_NAME,DOCTOR_NAME,DEPARTMENT_NAME,APPOINTMENT_DATE,BILL_AMOUNT,PAYMENT_STATUS
0,5001,Amit Sharma,Dr. Arvind Rao,Cardiology,2026-06-25,800,Paid
1,5002,Neha Verma,Dr. Meera Nair,Orthopedics,2026-06-25,700,Paid
2,5005,Karan Mehta,Dr. Vivek Malhotra,Cardiology,2026-06-27,900,Paid
3,5006,Sneha Joshi,Dr. Sandeep Khanna,General Medicine,2026-06-27,500,Paid
4,5008,Neha Verma,Dr. Arvind Rao,Cardiology,2026-06-28,800,Paid


## Step 8: Business question 2

Find total OPD revenue for paid bills only.

In [25]:
fetch_df("""
SELECT SUM(BILL_AMOUNT) AS TOTAL_OPD_REVENUE
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid'
""")

,TOTAL_OPD_REVENUE
0,3700


## Step 9: Business question 3

Find department-wise revenue.

In [26]:
fetch_df("""
SELECT
    DEPARTMENT_NAME,
    SUM(BILL_AMOUNT) AS TOTAL_REVENUE
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid'
GROUP BY DEPARTMENT_NAME
ORDER BY TOTAL_REVENUE DESC
""")

,DEPARTMENT_NAME,TOTAL_REVENUE
0,Cardiology,2500
1,Orthopedics,700
2,General Medicine,500


## Step 10: Business question 4

Find doctor-wise consultation revenue.

In [27]:
fetch_df("""
SELECT
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    COUNT(APPOINTMENT_ID) AS TOTAL_APPOINTMENTS,
    SUM(BILL_AMOUNT) AS TOTAL_REVENUE
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid'
GROUP BY DOCTOR_NAME, DEPARTMENT_NAME
ORDER BY TOTAL_REVENUE DESC
""")

,DOCTOR_NAME,DEPARTMENT_NAME,TOTAL_APPOINTMENTS,TOTAL_REVENUE
0,Dr. Arvind Rao,Cardiology,2,1600
1,Dr. Vivek Malhotra,Cardiology,1,900
2,Dr. Meera Nair,Orthopedics,1,700
3,Dr. Sandeep Khanna,General Medicine,1,500


## Step 11: Business question 5

Find all pending appointments.

In [28]:
fetch_df("""
SELECT
    APPOINTMENT_ID,
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    APPOINTMENT_DATE,
    APPOINTMENT_TIME
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE APPOINTMENT_STATUS = 'Pending'
""")

,APPOINTMENT_ID,PATIENT_NAME,DOCTOR_NAME,DEPARTMENT_NAME,APPOINTMENT_DATE,APPOINTMENT_TIME
0,5003,Rohan Gupta,Dr. Sandeep Khanna,General Medicine,2026-06-26,09:30:00
1,5007,Amit Sharma,Dr. Meera Nair,Orthopedics,2026-06-28,10:30:00


## Step 12: Business question 6

Find unpaid bills.

In [29]:
fetch_df("""
SELECT
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    BILL_AMOUNT,
    PAYMENT_STATUS
FROM V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Unpaid'
""")

,PATIENT_NAME,DOCTOR_NAME,DEPARTMENT_NAME,BILL_AMOUNT,PAYMENT_STATUS
0,Rohan Gupta,Dr. Sandeep Khanna,General Medicine,500,Unpaid
1,Amit Sharma,Dr. Meera Nair,Orthopedics,700,Unpaid


## Step 13: Business question 7

Find city-wise patient visits.

In [30]:
fetch_df("""
SELECT
    PATIENT_CITY,
    COUNT(APPOINTMENT_ID) AS TOTAL_NUMBER_OF_APPOINTMENTS
FROM V_OPD_APPOINTMENT_ANALYTICS
GROUP BY PATIENT_CITY
ORDER BY TOTAL_NUMBER_OF_APPOINTMENTS DESC, PATIENT_CITY
""")

,PATIENT_CITY,TOTAL_NUMBER_OF_APPOINTMENTS
0,Delhi,3
1,Ghaziabad,2
2,Noida,2
3,Faridabad,1


## Step 14: Validate department daily revenue view

This checks the final summary view required in the assignment.

In [31]:
fetch_df("""
SELECT *
FROM V_DEPARTMENT_DAILY_REVENUE
ORDER BY APPOINTMENT_DATE, DEPARTMENT_NAME
""")

,APPOINTMENT_DATE,DEPARTMENT_NAME,TOTAL_APPOINTMENTS,TOTAL_PAID_BILLS,TOTAL_REVENUE
0,2026-06-25,Cardiology,1,1,800
1,2026-06-25,Orthopedics,1,1,700
2,2026-06-27,Cardiology,1,1,900
3,2026-06-27,General Medicine,1,1,500
4,2026-06-28,Cardiology,1,1,800


## Step 15: Close the connection

Close database resources after execution.

In [32]:
cursor.close()
conn.close()
print("Connection closed")

Connection closed
